# EV Policy Assistant

## Stage 1: environment checks

Stage 1 setup checks. A separate Stage 2 worked example follows below. Adapted from the course repository at commit `33c2faa22450cde16ead9071f7ce7ecc78ca592a`: Lab 4 cells 6–8, 38, 65, 77, 106 and 150; Exercise 2 cell 4. AI assistance adapted these setup checks; no policy-answering pipeline is implemented here.

Run from the project folder with the project’s Python 3.12 environment. Put your Groq key in the local `.env` file. Notebook outputs should be cleared before committing.

In [ ]:
import os
import sys
import math
from importlib.metadata import version
from dotenv import load_dotenv
import gradio as gr
from langchain.chat_models import init_chat_model
from langchain.prompts import ChatPromptTemplate
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_community.document_loaders import UnstructuredPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(override=True)

assert sys.version_info[:2] == (3, 12), "Select the project Python 3.12 kernel."
print("Python:", sys.version.split()[0])
for package in ["langchain", "langchain-chroma", "langchain-ollama", "langchain-groq", "gradio", "pypdf", "unstructured"]:
    print(package, version(package))

### Local embeddings

Ollama must be running with `nomic-embed-text` available. This checks one query; it does not build an index.

In [ ]:
embeddings_model = OllamaEmbeddings(model="nomic-embed-text")
query_embedding = embeddings_model.embed_query("EV policy setup check")
assert len(query_embedding) > 0
assert all(math.isfinite(value) for value in query_embedding)
print("Embedding dimensions:", len(query_embedding))

### Groq connection

This sends a small test prompt to Groq and uses the account’s API allowance. A missing key stops the check; it is not a successful connection test.

In [ ]:
if not os.environ.get("GROQ_API_KEY"):
    raise ValueError("Add GROQ_API_KEY to the local .env file and rerun the setup cells.")

model_name = "openai/gpt-oss-120b"
llm = init_chat_model(model_name, model_provider="groq", temperature=0, max_tokens=256, timeout=30, max_retries=0)
response = llm.invoke("Reply with only OK.")
assert response.content, "Groq returned an empty response."
print(response.content)

## Stage 2: page-loading example

AI-assisted worked example for one page, within the agreed helper-only scope. It starts Stage 2; it is not the completed ingestion pipeline.

Reuses Exercise 2 cell 9's `Document(page_content=..., metadata=...)` pattern and Lab 4 cells 65–66's load-then-inspect sequence at course commit `33c2faa22450cde16ead9071f7ce7ecc78ca592a`. `pypdf.PdfReader` is an added page-preserving alternative, not the loader used in the course. Physical page 18 is Python page index 17. `conditions_pdf_page` links to page 19 of the same source; page 18 remains the citation for this text.

These cells run independently of the model setup above: no API key, Groq call or Ollama request is needed. Original PDFs remain the citation targets. Manual review is deferred; the example retains the manifest's unverified status.


In [ ]:
import json
import hashlib
from pathlib import Path
from pypdf import PdfReader
from langchain_core.documents import Document

source_manifest = json.loads(Path('data/source_manifest.json').read_text())
policy_source = next(source for source in source_manifest['sources']
                     if source['source_id'] == 'maharashtra_policy_2025-05-23')
policy_path = Path(policy_source['filename'])
assert hashlib.sha256(policy_path.read_bytes()).hexdigest() == policy_source['sha256']

pdf_page = 18
policy_text = PdfReader(policy_path).pages[pdf_page - 1].extract_text()
if not policy_text or not policy_text.strip():
    raise ValueError(f'No readable text on PDF page {pdf_page}.')

policy_document = Document(page_content=policy_text, metadata={
    'state': policy_source['state'],
    'policy_year': policy_source['policy_year'],
    'source_id': policy_source['source_id'],
    'source': policy_source['filename'],
    'document_title': policy_source['document_title'],
    'official_url': policy_source['official_url'],
    'document_date': policy_source['document_date'],
    'pdf_page': pdf_page,
    'conditions_pdf_page': 19,
    'team_verified': policy_source['team_verified'],
    'accepted_for_ingestion': policy_source['accepted_for_ingestion'],
    'current_benefit_availability': policy_source['current_benefit_availability'],
    'current_entitlement_answers_allowed': policy_source['current_entitlement_answers_allowed'],
    'verification_cutoff': source_manifest['verification_cutoff'],
})

print(policy_document.metadata)
print(policy_document.page_content)


In [ ]:
assert policy_document.metadata['pdf_page'] == 18
assert policy_document.metadata['conditions_pdf_page'] == 19
assert policy_document.metadata['team_verified'] is False
assert policy_document.metadata['accepted_for_ingestion'] is False
assert policy_document.metadata['current_entitlement_answers_allowed'] is False
assert 'Table 2: Demand Incentives for EVs' in policy_document.page_content
assert 'Maximum' in policy_document.page_content and 'Rupees' in policy_document.page_content
print('One candidate page loaded with provenance. Stage 2 is still in progress.')


### Load the English policy pages

This bounded extension of the worked example creates ten page records. It uses Exercise 2 cell 9's list/`Document` pattern with the page-preserving reader above. Run the Stage 2 example first; the Stage 1 model cells are still unnecessary.

Copy the shared metadata, then set the physical page for each record. The conditions-page link belongs only to Table 2 on page 18; copying it onto every page would create a wrong relationship. OCR loading, other cross-page relationships and splitting remain the team's next implementation work.


In [ ]:
policy_reader = PdfReader(policy_path)
assert len(policy_reader.pages) == policy_source['pdf_page_count']

policy_metadata = policy_document.metadata.copy()
policy_metadata.pop('pdf_page')
policy_metadata.pop('conditions_pdf_page')

policy_documents = []
for pdf_page in policy_source['candidate_pdf_pages']:
    page_text = policy_reader.pages[pdf_page - 1].extract_text()
    if not page_text or not page_text.strip():
        raise ValueError(f'No readable text on PDF page {pdf_page}.')

    page_metadata = policy_metadata.copy()
    page_metadata['pdf_page'] = pdf_page
    if pdf_page == 18:
        page_metadata['conditions_pdf_page'] = 19

    policy_documents.append(Document(page_content=page_text, metadata=page_metadata))

print('English policy pages loaded:', len(policy_documents))
print('Physical pages:', [doc.metadata['pdf_page'] for doc in policy_documents])


In [ ]:
assert len(policy_documents) == 10
assert [doc.metadata['pdf_page'] for doc in policy_documents] == list(range(16, 26))
assert len({(doc.metadata['source_id'], doc.metadata['pdf_page'])
            for doc in policy_documents}) == 10
assert all(doc.page_content.strip() for doc in policy_documents)
assert all(doc.metadata['team_verified'] is False for doc in policy_documents)
assert all(doc.metadata['accepted_for_ingestion'] is False for doc in policy_documents)
assert all(doc.metadata['current_entitlement_answers_allowed'] is False
           for doc in policy_documents)
assert all(doc.metadata['current_benefit_availability'] == 'not_verified'
           for doc in policy_documents)
assert all('conditions_pdf_page' not in doc.metadata
           for doc in policy_documents if doc.metadata['pdf_page'] != 18)
assert policy_documents[2].page_content == policy_document.page_content
assert policy_documents[2].metadata == policy_document.metadata
print('Ten candidate page records checked. OCR pages and splitting remain pending.')


### Extend the example

Team implementation continues here:

1. The example above now loads the ten English pages. Check its metadata-copying logic before extending it to other document types.
2. Read `data/ocr/maharashtra/review.json` and load its eight explicit `proposed_text` files. Check their hashes and keep each record's source/page identity; a folder-wide text search would duplicate raw and proposed versions.
3. Check the 18 page records before splitting. Carry the July clarification and August old/replacement/distribution roles from the manifest into metadata.
4. Adapt Lab 4 cell 77's splitter. Its 1000-character size and 200 overlap preserve Table 2 in the preliminary check, but page-19 conditions need an explicit link to page 18. Check all chunks and cross-page clauses before accepting the settings.

Keep distribution-only text out of answer chunks and old wording distinguishable from its replacement. Manual source review and the team's two source-backed examples are deferred to the final review batch. See `STAGE_2_HANDOFF.md` for the complete technical checks. No index or answers are built in this stage.
